# NumCompute streaming demo

This working demo tests CSV chunking, incremental preprocessing, streaming statistics and evaluation metrics using `demo/data/sample.csv`. Model training, pipelines and metric plots will be added later.

## 0. Set up and run

Select a Python environment with NumPy and the Jupyter kernel installed, restart the kernel and choose **Run All**. Run from the project root or `demo/`.

The CSV contains 3,000 fictional records. `age` and `income` are numerical features; both contain missing tokens that the CSV reader converts to NaN. `city` contains some blank cells. Section 4 evaluates the existing `city_predict` simulation to test metric updates; model evaluation will use real tree predictions once the training demo is added.

We reuse the existing `numcompute.io` reader because `numcompute_stream.io` is still a placeholder. All preprocessing and streaming statistics below come from the new `numcompute_stream` package.

Each subsection includes a short comparison with the original `numcompute` API on the same data. Batch arrays are loaded only for comparison; streaming updates still read chunks independently.

In [1]:
import sys
from pathlib import Path
import numpy as np

root = Path.cwd() if (Path.cwd() / "numcompute").is_dir() else Path.cwd().parent
sys.path.insert(0, str(root))
sample_path = root / "demo" / "data" / "sample.csv"

from numcompute import io, preprocessing as batch_pp, stats as batch_stats, metrics as batch_metrics
from numcompute_stream import preprocessing as pp, stats, metrics

np.set_printoptions(precision=3, suppress=True)
# print("Sample file:", sample_path)

## 1. Read the CSV in chunks

Each chunk has shape `(rows, features)`. Here the features are `age` and `income`, and each chunk contains up to 200 rows. Blank cells, `NA`, `NaN`, `NULL` and `N/A` become numeric NaN.

`numeric_chunks()` opens a fresh reader each time, so each independent example starts at the beginning of the same CSV. The reader yields one chunk at a time instead of loading the entire file.

In [2]:
chunk_size = 200
feature_names = ["age", "income"]

def numeric_chunks():
    return io.iter_csv_chunks(
        sample_path, chunk_size=chunk_size, usecols=feature_names, dtype=float,
    )

first_chunk = next(numeric_chunks())
print("CSV columns:", io.read_header(sample_path))
print("Feature order:", feature_names)
print("First chunk shape:", first_chunk.shape)
print("Missing cells in the first chunk:", np.isnan(first_chunk).sum(axis=0))
print("First five rows:\n", first_chunk[:5])

CSV columns: ['age', 'income', 'city', 'city_predict']
Feature order: ['age', 'income']
First chunk shape: (200, 2)
Missing cells in the first chunk: [15 22]
First five rows:
 [[   56.   81699.13]
 [   50.   48741.14]
 [   56.   54486.45]
 [   35.   84968.44]
 [   70.   74190.3 ]]


In [3]:
n_chunks = 0
n_rows = 0
for chunk in numeric_chunks():
    n_chunks += 1
    n_rows += len(chunk)
    if n_chunks <= 2:
        print(f"Chunk {n_chunks}: {chunk.shape}")

print("Total chunks:", n_chunks)
print("Total rows:", n_rows)
assert n_chunks == 15 and n_rows == 3000

Chunk 1: (200, 2)
Chunk 2: (200, 2)
Total chunks: 15
Total rows: 3000


**Original NumCompute comparison**

In [4]:
# numcompute: read the same CSV in one call.
X_batch = io.load_csv(sample_path, usecols=feature_names, dtype=float)
X_chunked = np.vstack(list(numeric_chunks()))  # Comparison only.

print("Batch shape:", X_batch.shape)
print("Chunked shape:", X_chunked.shape)

same_data = np.array_equal(X_chunked, X_batch, equal_nan=True)
print("Same data:", same_data)
assert same_data
del X_chunked

Batch shape: (3000, 2)
Chunked shape: (3000, 2)
Same data: True


## 2. Incremental preprocessing (`preprocessing.py`)

`fit(X)` starts again using only X. `partial_fit(X_chunk)` keeps earlier observations and updates learned parameters with the new chunk. `transform(X)` applies the current parameters without learning. Every reduction below works separately on each feature (axis 0).

### 2.1 StandardScaler: update the mean and variance

The scaler uses Chan/Welford merging to maintain cumulative means and population variances (`ddof=0`). It weights observations correctly when chunk sizes or missing-value counts differ. `n_samples_seen_` counts valid observations per feature, so its values can differ from the number of received rows.

Watch the first two updates: each line describes **all data received so far**, not just the latest chunk. NaNs are ignored during learning and remain NaN during transformation. Constant columns use a scale of one.

In [5]:
scaler = pp.StandardScaler()
for number, chunk in enumerate(numeric_chunks(), start=1):
    scaler.partial_fit(chunk)
    if number <= 2:
        print(f"After chunk {number}: mean={scaler.mean_}, variance={scaler.var_}")

print("Final valid counts:", scaler.n_samples_seen_)
print("Final means:", scaler.mean_)
print("Final scales:", scaler.scale_)
print("First five rows with the final parameters:\n", scaler.transform(first_chunk[:5]))

After chunk 1: mean=[   42.135 83924.003], variance=[1.112e+02 5.064e+08]
After chunk 2: mean=[   42.406 85897.876], variance=[1.238e+02 1.136e+09]
Final valid counts: [2820 2760]
Final means: [   42.327 87823.689]
Final scales: [   11.149 42079.088]
First five rows with the final parameters:
 [[ 1.226 -0.146]
 [ 0.688 -0.929]
 [ 1.226 -0.792]
 [-0.657 -0.068]
 [ 2.482 -0.324]]


**Original NumCompute comparison**

In [6]:
# numcompute: fit once on all observations.
batch_scaler = batch_pp.StandardScaler().fit(X_batch)
print("Mean: streaming | numcompute")
print(np.column_stack((scaler.mean_, batch_scaler.mean_)))
print("Variance: streaming | numcompute")
print(np.column_stack((scaler.var_, batch_scaler.scale_ ** 2)))

np.testing.assert_allclose(scaler.mean_, batch_scaler.mean_)
np.testing.assert_allclose(scaler.var_, batch_scaler.scale_ ** 2)
same_scaled = np.allclose(scaler.transform(X_batch), batch_scaler.transform(X_batch), equal_nan=True)

print("Same transformed results:", same_scaled)
assert same_scaled

Mean: streaming | numcompute
[[   42.327    42.327]
 [87823.689 87823.689]]
Variance: streaming | numcompute
[[1.243e+02 1.243e+02]
 [1.771e+09 1.771e+09]]
Same transformed results: True


### 2.2 MinMaxScaler: expand the observed range

This scaler updates the smallest and largest values seen in each feature. After the full pass, the observed minima map to -1 and the maxima map to 1. Missing cells remain NaN. New values outside the learned range are not clipped; a later `partial_fit()` can expand that range.

In [7]:
minmax = pp.MinMaxScaler(feature_range=(-1, 1))
for number, chunk in enumerate(numeric_chunks(), start=1):
    minmax.partial_fit(chunk)
    if number <= 2:
        print(f"After chunk {number}: min={minmax.data_min_}, max={minmax.data_max_}")

print("Final minima:", minmax.data_min_)
print("Final maxima:", minmax.data_max_)
endpoints = np.vstack((minmax.data_min_, minmax.data_max_))
print("Transformed minima / maxima:\n", minmax.transform(endpoints))

After chunk 1: min=[   18.   33438.53], max=[    70.   139833.35]
After chunk 2: min=[   18.   33438.53], max=[    70. 450000.]
Final minima: [   18.   33020.42]
Final maxima: [    70. 690000.]
Transformed minima / maxima:
 [[-1. -1.]
 [ 1.  1.]]


**Original NumCompute comparison**

In [8]:
batch_minmax = batch_pp.MinMaxScaler(feature_range=(-1, 1)).fit(X_batch)
print("Minimum: streaming | numcompute")
print(np.column_stack((minmax.data_min_, batch_minmax.data_min_)))

print("Range: streaming | numcompute")
print(np.column_stack((minmax.data_range_, batch_minmax.data_range_)))

np.testing.assert_allclose(minmax.data_min_, batch_minmax.data_min_)
np.testing.assert_allclose(minmax.data_range_, batch_minmax.data_range_)

same_minmax = np.allclose(minmax.transform(X_batch), batch_minmax.transform(X_batch), equal_nan=True)
print("Same transformed results:", same_minmax)
assert same_minmax

Minimum: streaming | numcompute
[[   18.      18.  ]
 [33020.42 33020.42]]
Range: streaming | numcompute
[[    52.       52.  ]
 [656979.58 656979.58]]
Same transformed results: True


### 2.3 SimpleImputer: update mean fill estimates

The mean imputer learns only from original nonmissing cells. Missing cells do not count as observations, and calling `transform()` does not add filled values to the estimates. If a column has no observations yet, its fallback is zero by default.

The printed fill values change as more chunks arrive. The final transform below fills missing cells in the first chunk using the final estimates; it is a retrospective check, not a prediction-time evaluation.

In [9]:
imputer = pp.SimpleImputer(strategy="mean")
for number, chunk in enumerate(numeric_chunks(), start=1):
    imputer.partial_fit(chunk)
    if number <= 2:
        print(f"After chunk {number}: fill values={imputer.statistics_}")

filled_first = imputer.transform(first_chunk)
print("Final fill values:", imputer.statistics_)
print("Valid observations used:", imputer.n_samples_seen_)
print("Missing before:", np.isnan(first_chunk).sum(axis=0))
print("Missing after:", np.isnan(filled_first).sum(axis=0))

After chunk 1: fill values=[   42.135 83924.003]
After chunk 2: fill values=[   42.406 85897.876]
Final fill values: [   42.327 87823.689]
Valid observations used: [2820 2760]
Missing before: [15 22]
Missing after: [0 0]


**Original NumCompute comparison**

In [10]:
batch_imputer = batch_pp.SimpleImputer(strategy="mean").fit(X_batch)
print("Mean fill values: streaming | numcompute")
print(np.column_stack((imputer.statistics_, batch_imputer.statistics_)))
np.testing.assert_allclose(imputer.statistics_, batch_imputer.statistics_)
same_mean_fill = np.allclose(imputer.transform(X_batch), batch_imputer.transform(X_batch))
print("Same filled results:", same_mean_fill)
assert same_mean_fill

Mean fill values: streaming | numcompute
[[   42.327    42.327]
 [87823.689 87823.689]]
Same filled results: True


### 2.4 Median and constant imputation

Median imputation also updates across chunks. An exact cumulative median cannot be obtained by averaging chunk medians, so this implementation keeps private copies of previous chunks. Mean imputation stores only per-feature state; exact median imputation uses memory proportional to all received rows.

In [11]:
median_imputer = pp.SimpleImputer(strategy="median")
for chunk in numeric_chunks():
    median_imputer.partial_fit(chunk)

print("Final median fill values:", median_imputer.statistics_)
print("Missing after median filling:", np.isnan(median_imputer.transform(first_chunk)).sum())

Final median fill values: [   42.    82603.155]
Missing after median filling: 0


**Original NumCompute comparison**

In [12]:
batch_median = batch_pp.SimpleImputer(strategy="median").fit(X_batch)
print("Median fill values: streaming | numcompute")
print(np.column_stack((median_imputer.statistics_, batch_median.statistics_)))
np.testing.assert_allclose(median_imputer.statistics_, batch_median.statistics_)
same_median_fill = np.allclose(median_imputer.transform(X_batch), batch_median.transform(X_batch))
print("Same filled results:", same_median_fill)
assert same_median_fill

Median fill values: streaming | numcompute
[[   42.       42.   ]
 [82603.155 82603.155]]
Same filled results: True


A constant does not need a changing estimate. The same `partial_fit` interface still checks feature counts and tracks valid observations. This example fills blank **categorical input cells** with `Unknown`, matching the old demo. If city is later used as the prediction target, blank target rows must instead be excluded from supervised training and evaluation.

In [13]:
city_imputer = pp.SimpleImputer(fill_value="Unknown")
for city_chunk in io.iter_csv_chunks(sample_path, chunk_size=chunk_size, usecols="city"):
    city_imputer.partial_fit(city_chunk)

city_preview = next(io.iter_csv_chunks(sample_path, chunk_size=chunk_size, usecols="city"))
blank_row = np.flatnonzero(city_preview[:, 0] == "")[0]
examples = city_preview[[0, blank_row]]
print("Before:", examples.ravel())
print("After:", city_imputer.transform(examples).ravel())
print("Nonmissing cities received:", city_imputer.n_samples_seen_)

Before: ['Hobart' '']
After: ['Hobart' 'Unknown']
Nonmissing cities received: [2920]


**Original NumCompute comparison**

In [14]:
city_batch = io.load_csv(sample_path, usecols="city")
batch_city_imputer = batch_pp.SimpleImputer(fill_value="Unknown").fit(city_batch)
print("Streaming examples:", city_imputer.transform(examples).ravel())
print("numcompute examples:", batch_city_imputer.transform(examples).ravel())
same_city_fill = np.array_equal(city_imputer.transform(city_batch), batch_city_imputer.transform(city_batch))
print("Same filled results for all cities:", same_city_fill)
assert same_city_fill

Streaming examples: ['Hobart' 'Unknown']
numcompute examples: ['Hobart' 'Unknown']
Same filled results for all cities: True


### 2.5 OneHotEncoder: learn categories as they arrive

`partial_fit()` adds new nonmissing categories without forgetting earlier ones. For this example only, hold Sydney rows back so that a new category arrives later. All rows still come from the CSV; this rearrangement is just to make category expansion visible.

First, send the other cities in 200-row chunks. Sydney is unknown during this stage, so `handle_unknown="ignore"` encodes it as zeros. With `handle_unknown="error"`, transforming an unseen city would raise ValueError instead. Neither policy prevents `partial_fit()` from learning that city later.

In [15]:
encoder = pp.OneHotEncoder(handle_unknown="ignore")
initial_cities = city_batch[city_batch[:, 0] != "Sydney"]
later_cities = city_batch[city_batch[:, 0] == "Sydney"]

for start in range(0, len(initial_cities), chunk_size):
    encoder.partial_fit(initial_cities[start:start + chunk_size])

print("Categories before Sydney:", encoder.categories_[0])
print("Output columns before:", encoder.n_features_out_)
print("Sydney before learning:", encoder.transform(later_cities[:1]))

Categories before Sydney: ['Adelaide' 'Brisbane' 'Hobart' 'Melbourne' 'Perth']
Output columns before: 5
Sydney before learning: [[0. 0. 0. 0. 0.]]


Now add the Sydney rows as one later chunk. The output gains a column; blank city cells still encode to all zeros. Categories stay sorted, so adding a category can change column positions as well as width. Re-transform earlier raw rows with the current encoder before combining encodings from different stages.

In [16]:
encoder.partial_fit(later_cities)
blank_city = city_batch[city_batch[:, 0] == ""][:1]
print("Categories after Sydney:", encoder.categories_[0])
print("Output columns after:", encoder.n_features_out_)
print("Sydney after learning:", encoder.transform(later_cities[:1]))
print("Blank city encoding:", encoder.transform(blank_city))
assert encoder.n_features_out_ == 6
assert encoder.transform(later_cities[:1]).sum() == 1
assert encoder.transform(blank_city).sum() == 0

Categories after Sydney: ['Adelaide' 'Brisbane' 'Hobart' 'Melbourne' 'Perth' 'Sydney']
Output columns after: 6
Sydney after learning: [[0. 0. 0. 0. 0. 1.]]
Blank city encoding: [[0. 0. 0. 0. 0. 0.]]


**Original NumCompute comparison**

In [17]:
batch_encoder = batch_pp.OneHotEncoder(handle_unknown="ignore").fit(city_batch)
print("Streaming categories:", encoder.categories_[0])
print("numcompute categories:", batch_encoder.categories_[0])
same_encoding = np.array_equal(encoder.transform(city_batch), batch_encoder.transform(city_batch))
print("Same encoded results for all rows:", same_encoding)
assert same_encoding

Streaming categories: ['Adelaide' 'Brisbane' 'Hobart' 'Melbourne' 'Perth' 'Sydney']
numcompute categories: ['Adelaide' 'Brisbane' 'Hobart' 'Melbourne' 'Perth' 'Sydney']
Same encoded results for all rows: True


## 3. Streaming statistics (`stats.py`)

### 3.1 Update cumulative moments

`RunningStats.update(chunk)` merges each new chunk into the same accumulator. `n` includes all received rows; `count_` counts nonmissing cells separately per feature. Mean, variance, standard deviation, minimum and maximum each return shape `(2,)`, in age/income order. NaNs are ignored per column by default, without dropping entire rows.

Here `var()` and `std()` use population statistics. Pass `ddof=1` when sample variance or standard deviation is needed.

In [18]:
running = stats.RunningStats()
for number, chunk in enumerate(numeric_chunks(), start=1):
    running.update(chunk)
    if number <= 2:
        print(f"After chunk {number}: rows={running.n}, valid={running.count_}, mean={running.mean}")

print("Final rows and valid counts:", running.n, running.count_)
print("Mean:", running.mean)
print("Population variance:", running.var())
print("Population std:", running.std())
print("Min / max:", running.min, running.max)

After chunk 1: rows=200, valid=[185 178], mean=[   42.135 83924.003]
After chunk 2: rows=400, valid=[379 359], mean=[   42.406 85897.876]
Final rows and valid counts: 3000 [2820 2760]
Mean: [   42.327 87823.689]
Population variance: [1.243e+02 1.771e+09]
Population std: [   11.149 42079.088]
Min / max: [   18.   33020.42] [    70. 690000.]


**Original NumCompute comparison**

In [19]:
# Match the new per-column NaN handling with the old batch functions.
batch_mean = batch_stats.mean(X_batch, axis=0, skip_nan=True)
batch_std = batch_stats.std(X_batch, axis=0, ddof=0, skip_nan=True)
batch_min = batch_stats.min(X_batch, axis=0, skip_nan=True)
batch_max = batch_stats.max(X_batch, axis=0, skip_nan=True)
batch_summary = np.array([batch_mean, batch_std ** 2, batch_std, batch_min, batch_max])
stream_summary = np.array([running.mean, running.var(), running.std(), running.min, running.max])
print("Rows: mean, variance, std, min, max; columns: age, income")
print("Streaming:\n", stream_summary)
print("numcompute:\n", batch_summary)
print("All statistics match:", np.allclose(stream_summary, batch_summary))
np.testing.assert_allclose(stream_summary, batch_summary)

Rows: mean, variance, std, min, max; columns: age, income
Streaming:
 [[4.233e+01 8.782e+04]
 [1.243e+02 1.771e+09]
 [1.115e+01 4.208e+04]
 [1.800e+01 3.302e+04]
 [7.000e+01 6.900e+05]]
numcompute:
 [[4.233e+01 8.782e+04]
 [1.243e+02 1.771e+09]
 [1.115e+01 4.208e+04]
 [1.800e+01 3.302e+04]
 [7.000e+01 6.900e+05]]
All statistics match: True


### 3.2 Query exact quantiles and an income histogram

With the default `store_data=True`, the accumulator retains incoming data for exact distribution queries. These queries recalculate results from the accumulated history; they are not constant-memory approximations.

Quantile probabilities range from 0 to 1. With `axis=0`, three probabilities return shape `(3, 2)`; `axis=None` instead combines all cells, which is usually inappropriate for age and income because their units differ. The histogram uses only income (`feature=1`), ignores missing incomes and includes the large income outliers in its range.

In [20]:
print("25th / 50th / 75th percentiles; columns are age and income:")
print(running.quantile([0.25, 0.5, 0.75], axis=0))
print("Median:", running.median())

hist_counts, hist_edges = running.histogram(bins=6, feature=1)
print("Income bin edges:", hist_edges)
print("Income bin counts:", hist_counts)
print("Histogram observations:", hist_counts.sum())
assert hist_counts.sum() == running.count_[1]

25th / 50th / 75th percentiles; columns are age and income:
[[    34.     66329.5  ]
 [    42.     82603.155]
 [    50.    104537.24 ]]
Median: [   42.    82603.155]
Income bin edges: [ 33020.42  142517.017 252013.613 361510.21  471006.807 580503.403
 690000.   ]
Income bin counts: [2740    2    3    6    3    6]
Histogram observations: 2760


**Original NumCompute comparison**

In [21]:
batch_quartiles = batch_stats.quantile(X_batch, [0.25, 0.5, 0.75], axis=0, skip_nan=True)
batch_medians = batch_stats.median(X_batch, axis=0, skip_nan=True)
batch_hist, batch_edges = batch_stats.histogram(X_batch[:, 1], bins=6)
print("numcompute quartiles:\n", batch_quartiles)
print("Median: streaming | numcompute")
print(np.column_stack((running.median(), batch_medians)))
print("Income bin counts: streaming | numcompute")
print(np.column_stack((hist_counts, batch_hist)))
np.testing.assert_allclose(running.quantile([0.25, 0.5, 0.75]), batch_quartiles)
np.testing.assert_allclose(running.median(), batch_medians)
np.testing.assert_array_equal(hist_counts, batch_hist)
np.testing.assert_allclose(hist_edges, batch_edges)
print("Quantiles, medians, histogram counts and edges match.")

numcompute quartiles:
 [[    34.     66329.5  ]
 [    42.     82603.155]
 [    50.    104537.24 ]]
Median: streaming | numcompute
[[   42.       42.   ]
 [82603.155 82603.155]]
Income bin counts: streaming | numcompute
[[2740 2740]
 [   2    2]
 [   3    3]
 [   6    6]
 [   3    3]
 [   6    6]]
Quantiles, medians, histogram counts and edges match.


### 3.3 Keep only moments, reuse update_stats, and reset

If quantiles and histograms are unnecessary, `store_data=False` keeps only per-feature running state. `update_stats(chunk, accumulator)` updates that same object; with no accumulator, it creates a new one.

`reset()` clears the current stream. `partial_fit()` is another name for `update()`, so statistics can follow the same calling convention as preprocessors. Distribution queries require history storage and are unavailable in moments-only mode.

In [22]:
moments_only = stats.RunningStats(store_data=False)
for chunk in numeric_chunks():
    stats.update_stats(chunk, moments_only)

moments_full_mean = moments_only.mean.copy()
print("Mean without storing history:", moments_full_mean)
np.testing.assert_allclose(moments_only.mean, running.mean)

moments_only.reset()
print("Rows after reset:", moments_only.n)
moments_only.partial_fit(first_chunk)
print("Rows after one new partial_fit:", moments_only.n)

Mean without storing history: [   42.327 87823.689]
Rows after reset: 0
Rows after one new partial_fit: 200


**Original NumCompute comparison**

In [23]:
print("Mean after all chunks: moments-only | numcompute")
print(np.column_stack((moments_full_mean, batch_stats.mean(X_batch, axis=0, skip_nan=True))))
# The moments-only full-stream result was checked before reset above.
print("Mean after reset and one chunk: streaming | numcompute")
first_batch_mean = batch_stats.mean(first_chunk, axis=0, skip_nan=True)
print(np.column_stack((moments_only.mean, first_batch_mean)))
np.testing.assert_allclose(moments_only.mean, first_batch_mean)
print("Same result after starting a new stream:", np.allclose(moments_only.mean, first_batch_mean))

Mean after all chunks: moments-only | numcompute
[[   42.327    42.327]
 [87823.689 87823.689]]
Mean after reset and one chunk: streaming | numcompute
[[   42.135    42.135]
 [83924.003 83924.003]]
Same result after starting a new stream: True


## 4. Streaming evaluation (`metrics.py`)

The original NumCompute functions and input checks are retained. The new classes add accumulated confusion counts, weighted chunk errors or score history around the same metric calculations.

Metrics use `update(y_true_chunk, y_pred_chunk)` to receive new observations, `result()` to read the current score and `reset()` to start a new evaluation. Results are calculated from samples, not averages of chunk scores. Empty updates do nothing; querying an empty metric raises RuntimeError.

Here `city` is the true target and `city_predict` is the existing simulated prediction. Exclude blank true cities before updating metrics. No model training is performed in this section.

### 4.1 Accumulate accuracy, precision, recall and F1

Accuracy measures the fraction of correct predictions. Precision measures how often a predicted class is correct; recall measures how many true examples of that class are found; F1 balances the two. Use `average="macro"` for these six cities so each observed class contributes equally. Other options are `"binary"`, `"micro"`, `"weighted"` and `None` (one score per class in `labels_` order). `zero_division=0` is the default for an undefined ratio.

In [24]:
def label_chunks():
    for chunk in io.iter_csv_chunks(
        sample_path, chunk_size=chunk_size, usecols=["city", "city_predict"],
    ):
        labelled = chunk[:, 0] != ""
        yield chunk[labelled, 0], chunk[labelled, 1]

scores = {
    "accuracy": metrics.Accuracy(),
    "precision": metrics.Precision(average="macro"),
    "recall": metrics.Recall(average="macro"),
    "f1": metrics.F1(average="macro"),
}

In [25]:
for number, (true_chunk, pred_chunk) in enumerate(label_chunks(), start=1):
    for metric in scores.values():
        metric.update(true_chunk, pred_chunk)
    if number <= 2:
        print(f"After chunk {number}: cumulative accuracy={scores['accuracy'].result():.3f}")

print("Labelled samples received:", scores["accuracy"].n_samples_seen_)
for name, metric in scores.items():
    print(f"Final {name}: {metric.result():.3f}")

After chunk 1: cumulative accuracy=0.708
After chunk 2: cumulative accuracy=0.721
Labelled samples received: 2920
Final accuracy: 0.743
Final precision: 0.754
Final recall: 0.743
Final f1: 0.746


**Original NumCompute comparison**

In [26]:
label_data = io.load_csv(sample_path, usecols=["city", "city_predict"])
labelled = label_data[:, 0] != ""
y_true, y_pred = label_data[labelled, 0], label_data[labelled, 1]
batch_scores = {
    "accuracy": batch_metrics.accuracy(y_true, y_pred),
    "precision": batch_metrics.precision(y_true, y_pred, average="macro"),
    "recall": batch_metrics.recall(y_true, y_pred, average="macro"),
    "f1": batch_metrics.f1(y_true, y_pred, average="macro"),
}
print("Metric | streaming | numcompute")
for name, metric in scores.items():
    print(f"{name}: {metric.result():.6f} | {batch_scores[name]:.6f}")
    np.testing.assert_allclose(metric.result(), batch_scores[name])
print("All cumulative classification scores match.")

Metric | streaming | numcompute
accuracy: 0.743151 | 0.743151
precision: 0.753808 | 0.753808
recall: 0.743255 | 0.743255
f1: 0.746301 | 0.746301
All cumulative classification scores match.


### 4.2 Accumulate the confusion matrix

Rows represent true cities and columns predicted cities. Diagonal counts are correct predictions; off-diagonal counts show which cities are confused. `labels_` gives the order. New classes expand the accumulated matrix without losing earlier counts. Supply `labels=[...]` to keep a fixed order, including classes not yet observed.

In [27]:
confusion = metrics.ConfusionMatrix()
for true_chunk, pred_chunk in label_chunks():
    confusion.update(true_chunk, pred_chunk)

print("City order:", confusion.labels_)
print("Cumulative confusion matrix:\n", confusion.result())
print("Matrix total:", confusion.result().sum())
assert confusion.result().sum() == len(y_true)

City order: ['Adelaide' 'Brisbane' 'Hobart' 'Melbourne' 'Perth' 'Sydney']
Cumulative confusion matrix:
 [[352  68   9  56   0   3]
 [ 75 339  32  27  12   3]
 [  2  82 400   2   0   3]
 [ 14  30   0 358  43  42]
 [  0  40   8  57 341  42]
 [  0  15   0  52  33 380]]
Matrix total: 2920


**Original NumCompute comparison**

In [28]:
batch_confusion = batch_metrics.confusion_matrix(y_true, y_pred, labels=confusion.labels_)
print("numcompute confusion matrix:\n", batch_confusion)
print("Same confusion counts:", np.array_equal(confusion.result(), batch_confusion))
np.testing.assert_array_equal(confusion.result(), batch_confusion)

numcompute confusion matrix:
 [[352  68   9  56   0   3]
 [ 75 339  32  27  12   3]
 [  2  82 400   2   0   3]
 [ 14  30   0 358  43  42]
 [  0  40   8  57 341  42]
 [  0  15   0  52  33 380]]
Same confusion counts: True


### 4.3 Evaluate only the latest 500 labelled samples

Set `window_size=500` to keep a rolling sample window. It can cut through a chunk: this means the latest 500 accepted label pairs, not 500 raw CSV rows or a fixed number of chunks. `n_samples_seen_` counts all accepted pairs, while `n_samples_` is the current window count. Compare these recent scores with the cumulative scores to see how recent performance differs from the overall result. Every stateful metric in this module supports window_size.

In [29]:
rolling_accuracy = metrics.Accuracy(window_size=500)
rolling_f1 = metrics.F1(average="macro", window_size=500)
for number, (true_chunk, pred_chunk) in enumerate(label_chunks(), start=1):
    rolling_accuracy.update(true_chunk, pred_chunk)
    rolling_f1.update(true_chunk, pred_chunk)
    if number in (1, 2, n_chunks):
        print(f"Chunk {number}: recent accuracy={rolling_accuracy.result():.3f}, F1={rolling_f1.result():.3f}")

print("Active / total labelled samples:", rolling_accuracy.n_samples_, rolling_accuracy.n_samples_seen_)
print("Cumulative / recent accuracy:", scores["accuracy"].result(), rolling_accuracy.result())

Chunk 1: recent accuracy=0.708, F1=0.720
Chunk 2: recent accuracy=0.721, F1=0.727
Chunk 15: recent accuracy=0.780, F1=0.778
Active / total labelled samples: 500 2920
Cumulative / recent accuracy: 0.7431506849315068 0.78


**Original NumCompute comparison**

In [30]:
recent_accuracy = batch_metrics.accuracy(y_true[-500:], y_pred[-500:])
recent_f1 = batch_metrics.f1(y_true[-500:], y_pred[-500:], average="macro")
print("Accuracy: rolling | numcompute", rolling_accuracy.result(), recent_accuracy)
print("F1: rolling | numcompute", rolling_f1.result(), recent_f1)
np.testing.assert_allclose(rolling_accuracy.result(), recent_accuracy)
np.testing.assert_allclose(rolling_f1.result(), recent_f1)

Accuracy: rolling | numcompute 0.78 0.78
F1: rolling | numcompute 0.7775409539655187 0.7775409539655187


### 4.4 Accumulate binary ROC and AUC

ROC needs a score for the positive class, rather than a predicted city label. For this test, define **Sydney versus other cities** and use observed income as a simple illustrative score: larger income means a higher Sydney score. This is a heuristic, not a trained probability model. Exclude missing income and blank city labels; all remaining rows come from the CSV.

ROC reports true/false positive rates across thresholds; AUC summarizes the ranking. Perfect ranking gives 1, reversed ranking gives 0, and equal scores give 0.5. Exact cumulative ROC/AUC retain all accepted scores so cross-chunk ties are handled together. A window bounds storage. A single-class chunk can be received, but result() needs both positive and negative samples in the active data.

In [31]:
usable = (city_batch[:, 0] != "") & ~np.isnan(X_batch[:, 1])
binary_true = (city_batch[usable, 0] == "Sydney").astype(int)
income_scores = X_batch[usable, 1]
roc = metrics.ROCCurve(pos_label=1)
roc_area = metrics.AUC(pos_label=1)
for start in range(0, len(binary_true), chunk_size):
    target = binary_true[start:start + chunk_size]
    score = income_scores[start:start + chunk_size]
    roc.update(target, score)
    roc_area.update(target, score)

fpr, tpr, thresholds = roc.result()
print("Accepted binary samples:", roc.n_samples_seen_)
print("Income-score AUC:", roc_area.result())
print("First five FPR / TPR points:\n", np.column_stack((fpr[:5], tpr[:5])))

Accepted binary samples: 2689
Income-score AUC: 0.9379131954722938
First five FPR / TPR points:
 [[0.    0.   ]
 [0.    0.007]
 [0.001 0.007]
 [0.003 0.007]
 [0.004 0.007]]


**Original NumCompute comparison**

In [32]:
batch_roc = batch_metrics.roc_curve(binary_true, income_scores, pos_label=1)
batch_auc = batch_metrics.auc(batch_roc[0], batch_roc[1])
print("AUC: streaming | numcompute", roc_area.result(), batch_auc)
for streamed, reference in zip(roc.result(), batch_roc):
    np.testing.assert_allclose(streamed, reference)
np.testing.assert_allclose(roc_area.result(), batch_auc)
print("ROC points, thresholds and AUC match.")

AUC: streaming | numcompute 0.9379131954722938 0.9379131954722938
ROC points, thresholds and AUC match.


### 4.5 Accumulate regression MSE

For a numerical check, predict the dataset's median age for each observed age. This constant baseline is used only to test the error accumulator. MSE is the average squared difference between the actual and predicted ages. Its cumulative update weights chunks by sample count; window_size can restrict it to recent samples.

In [33]:
observed_ages = X_batch[~np.isnan(X_batch[:, 0]), 0]
age_predictions = np.full_like(observed_ages, median_imputer.statistics_[0])
age_error = metrics.MSE()
for start in range(0, len(observed_ages), chunk_size):
    age_error.update(observed_ages[start:start + chunk_size], age_predictions[start:start + chunk_size])

print("Age baseline MSE:", age_error.result())
print("Observed ages evaluated:", age_error.n_samples_seen_)

Age baseline MSE: 124.41347517730496
Observed ages evaluated: 2820


**Original NumCompute comparison**

In [34]:
batch_error = batch_metrics.mse(observed_ages, age_predictions)
print("MSE: streaming | numcompute", age_error.result(), batch_error)
np.testing.assert_allclose(age_error.result(), batch_error)

MSE: streaming | numcompute 124.41347517730496 124.41347517730496


### 4.6 Reset before starting a new evaluation

Reading result() preserves state. reset() clears observations but keeps options such as window_size. After reset, receive a new nonempty update before querying a result.

In [35]:
rolling_accuracy.reset()
print("Samples after reset:", rolling_accuracy.n_samples_seen_)
rolling_accuracy.update(y_true[:10], y_pred[:10])
print("New evaluation accuracy:", rolling_accuracy.result())
print("numcompute on the same ten rows:", batch_metrics.accuracy(y_true[:10], y_pred[:10]))
np.testing.assert_allclose(rolling_accuracy.result(), batch_metrics.accuracy(y_true[:10], y_pred[:10]))
print("All streaming metric comparisons passed.")

Samples after reset: 0
New evaluation accuracy: 0.9
numcompute on the same ten rows: 0.9
All streaming metric comparisons passed.
